# Qwen-Image-2.1 Uncensored — Colab WebUI

在 Colab 上部署 **Qwen-Image-2.1 Uncensored** 的 GGUF 量化模型，并启动一个自定义 WebUI。

### 使用前：选择 L4 GPU

`运行时` → `更改运行时类型` → 硬件加速器选择 **L4 GPU**（24 GB 显存），然后点击 `全部运行`。

### 显存与量化的对应关系

| 显存 | DiT 量化 | 文本编码器 |
| --- | --- | --- |
| ≥ 40 GB | BF16 (14.23 GB) | bf16 |
| **20–40 GB（L4）** | **Q8_0 (7.59 GB)** | **int8** |
| 14–20 GB | Q6_K (5.88 GB) | int8 |
| 10–14 GB | Q5_K_M (5.22 GB) | int8 |
| < 10 GB | Q4_K_M (4.60 GB) | int8 |

脚本会自动读取显卡显存并选择最合适的量化：**L4 上是 Q8_0**。
上游作者推荐的 Q4_K_M 是给 8–12 GB 消费级显卡用的；在 24 GB 的 L4 上没有必要牺牲画质。

> 网页文件（`index.html` / `style.css` / `app.js`）全部存放在 GitHub 仓库中，由服务端在运行时拉取，**没有使用 base64 内联**。


In [ ]:
# 确认显卡型号与显存
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 从 GitHub 拉取部署脚本并执行（首次约 8-15 分钟，主要耗时在下载 ~17 GB 权重）
import os, urllib.request

REPO = "hsgwktb/qwen-image-2.1-webui"
REF  = "main"

os.environ["GITHUB_REPO"] = REPO
os.environ["GITHUB_REF"]  = REF
# os.environ["QWEN_QUANT"] = "Q6_K"   # 需要手动指定量化时取消注释

url = f"https://raw.githubusercontent.com/{REPO}/{REF}/colab_setup.py"
urllib.request.urlretrieve(url, "/content/colab_setup.py")
print("已下载 colab_setup.py，开始部署…\n")
!python /content/colab_setup.py

In [ ]:
# 获取 WebUI 的访问地址
from google.colab.output import eval_js

URL = eval_js("google.colab.kernel.proxyPort(7860)")
print("WebUI 地址：", URL)
print("(需要登录同一个 Google 账号才能访问)")

In [ ]:
# 可选：同时打开 ComfyUI 原生界面（自带节点编辑器）
from google.colab.output import eval_js

print("ComfyUI 原生界面：", eval_js("google.colab.kernel.proxyPort(8188)"))

## 说明

- **GPU**：必须选择 L4 / 其他 CUDA 显卡；CPU 运行时无法加载模型。
- **首次启动**：需要下载约 17 GB 权重（DiT 7.6 GB + 文本编码器 9.4 GB + VAE 0.7 GB），完成后缓存在 `/content`。
- **`/content` 是临时的**：运行时重置后需要重新下载。
- **切换画质/显存**：设置 `QWEN_QUANT`（`BF16` / `Q8_0` / `Q6_K` / `Q5_K_M` / `Q4_K_M` / `Q4_0`）后重跑。
- **显存不足**：脚本在显存 < 12 GB 时自动加 `--lowvram`；也可在 `colab_setup.py` 中把 `start_comfyui(lowvram=True)` 写死。
- **日志**：`/content/comfyui.log` 与 `/content/webui.log`。

### 前端资源来自 GitHub

`server.py` 启动时会从 `raw.githubusercontent.com` 拉取 `web/` 下的三个文件并转发（默认 `ASSET_SOURCE=proxy`）。
如果想在浏览器里直接引用 GitHub 地址，在部署前设置 `os.environ["ASSET_SOURCE"] = "github"`；
当时会改成由页面直接 `<link>` / `<script>` 指向 jsDelivr / GitHub raw。

### 内容说明

该模型没有内置安全过滤器，输出完全取决于提示词。请自行确认在你所在地区合法合规，并对生成内容负责。
